# **Challenge Rhine Freight**
The Rhine freight Challenge created by an European energy company that manufactures, stores and distributes conventional fuels and sustainable energies and services.
-   The Rhine river is one the most frequented inland waterways. It is the most
important route from the Amsterdam-Rotterdam-Antwerp (ARA) region to
Germany or Switzerland:
    -   Its fleet is estimated at about 6900 vessels.
    -   The volumes transported amount to about 200 million tonnes at the
    -   German-Dutch border, and about 310 million tonnes on the Dutch section of the river.
-   One of the components of the transportation costs is the freight price.
-   The freight price has an important influence on many operations. It affects:
    -   How Logistics book barges for transportation.
    -   Trading decisions, since the freight prices are major drivers for product prices in Germany, for example.



The freight price is characterized by having some extreme periods in which the price increases significantly.
-   Thebusiness is very interested in better understand and predict the extreme periods, because that’s where more opportunities lie.
-   Thebusiness believes that the freight price depends on:
    -   The Rhine water levels: Low water levels on the choke points impose a limit on the number of vessels that can go through
and on the amount of cargo each vessel can carry.
    -   Supply and demand

## **Challenge: What is the freight price, 5 business days ahead?**

### 1. Imports

In [24]:
import pandas as pd
import numpy as np
from pathlib import Path


### 2. Download datasets 

In [3]:
rf = pd.read_csv("RhineFreightData.csv", sep=";")
w = pd.read_csv("WeatherDataBasel.csv", sep=";")

2.1 Rhine Freight dataset

In [7]:
rf

,Date,Freight,Mogas_Flat,Brent_flat,Diesel_Flat,Heating_Flat,Kaub_level,freight_ar
0,03/01/2013,44.55,573.622200,260.662494,572.019772,554.576878,430.0,NaN
1,04/01/2013,41.55,577.435102,260.252075,574.219032,557.993052,389.0,NaN
2,07/01/2013,41.55,576.174887,258.999923,578.355303,561.713421,327.0,NaN
3,08/01/2013,40.05,577.156585,259.078404,579.239037,562.503444,329.0,NaN
4,09/01/2013,38.55,578.066208,259.375000,583.323044,567.144726,320.0,NaN
...,...,...,...,...,...,...,...,...
1708,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1709,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1710,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1711,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
print(f"Dataset shape: {rf.shape}")

Dataset shape: (1713, 8)


In [11]:
rf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1713 entries, 0 to 1712
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          1469 non-null   object 
 1   Freight       1454 non-null   float64
 2   Mogas_Flat    1469 non-null   float64
 3   Brent_flat    1469 non-null   float64
 4   Diesel_Flat   1469 non-null   float64
 5   Heating_Flat  1469 non-null   float64
 6   Kaub_level    1456 non-null   float64
 7   freight_ar    723 non-null    float64
dtypes: float64(7), object(1)
memory usage: 107.2+ KB


In [13]:
print(f"Number of duplicate rows: {rf.duplicated().sum()}")

Number of duplicate rows: 243


In [16]:
rf.isna().sum()

Date            244
Freight         259
Mogas_Flat      244
Brent_flat      244
Diesel_Flat     244
Heating_Flat    244
Kaub_level      257
freight_ar      990
dtype: int64

In [23]:
missing_pct = (rf.isna().mean()*100).round(2)
print(missing_pct.sort_values(ascending=False))

freight_ar      57.79
Freight         15.12
Kaub_level      15.00
Date            14.24
Brent_flat      14.24
Mogas_Flat      14.24
Heating_Flat    14.24
Diesel_Flat     14.24
dtype: float64


In [18]:
rf.nunique()

Date            1469
Freight          203
Mogas_Flat      1469
Brent_flat      1469
Diesel_Flat     1469
Heating_Flat    1468
Kaub_level       406
freight_ar       284
dtype: int64

2.2 Weather Dataset

In [5]:
w

,date,avg_uv_index,avgtemp,cloudcover,dewpoint,feelslike,heatindex,humidity,maxtemp,mintemp,...,totalsnow,uv_index,visibility,weather_code,weather_descriptions,wind_degree,wind_dir,wind_speed,windchill,windgust
0,01/01/2009,1.0,-1.0,17.0,-4.0,-6.0,-3.0,95.0,2.0,-5.0,...,0.0,1.0,7.0,113.0,['Clear'],48.0,NE,10.0,-6.0,19.0
1,02/01/2009,1.0,-3.0,63.0,-5.0,-9.0,-3.0,88.0,-2.0,-4.0,...,0.0,1.0,10.0,119.0,['Cloudy'],54.0,NE,17.0,-9.0,26.0
2,03/01/2009,1.0,-7.0,7.0,-12.0,-12.0,-9.0,78.0,-4.0,-11.0,...,0.0,1.0,10.0,113.0,['Clear'],67.0,ENE,6.0,-12.0,13.0
3,04/01/2009,1.0,-8.0,19.0,-12.0,-10.0,-8.0,74.0,-4.0,-10.0,...,0.0,1.0,10.0,116.0,['Partly cloudy'],264.0,W,4.0,-10.0,9.0
4,05/01/2009,1.0,-7.0,74.0,-8.0,-11.0,-6.0,89.0,-5.0,-8.0,...,0.0,1.0,10.0,122.0,['Overcast'],56.0,NE,10.0,-11.0,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
print(f"Dataset shape: {w.shape}")

Dataset shape: (4017, 25)


In [12]:
w.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4017 entries, 0 to 4016
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  3652 non-null   object 
 1   avg_uv_index          3652 non-null   float64
 2   avgtemp               3652 non-null   float64
 3   cloudcover            3652 non-null   float64
 4   dewpoint              3652 non-null   float64
 5   feelslike             3652 non-null   float64
 6   heatindex             3652 non-null   float64
 7   humidity              3652 non-null   float64
 8   maxtemp               3652 non-null   float64
 9   mintemp               3652 non-null   float64
 10  moon_illumination     3652 non-null   float64
 11  moon_phase            3652 non-null   object 
 12  precip                3652 non-null   float64
 13  pressure              3652 non-null   float64
 14  sunhour               3652 non-null   float64
 15  totalsnow            

In [15]:
print(f"Number of duplicate rows: {w.duplicated().sum()}")

Number of duplicate rows: 364


In [17]:
w.isna().sum()

date                    365
avg_uv_index            365
avgtemp                 365
cloudcover              365
dewpoint                365
feelslike               365
heatindex               365
humidity                365
maxtemp                 365
mintemp                 365
moon_illumination       365
moon_phase              365
precip                  365
pressure                365
sunhour                 365
totalsnow               365
uv_index                365
visibility              365
weather_code            365
weather_descriptions    365
wind_degree             365
wind_dir                365
wind_speed              365
windchill               365
windgust                365
dtype: int64

In [22]:
missing_pct = (w.isna().mean()*100).round(2)
print(missing_pct.sort_values(ascending=False))

date                    9.09
avg_uv_index            9.09
avgtemp                 9.09
cloudcover              9.09
dewpoint                9.09
feelslike               9.09
heatindex               9.09
humidity                9.09
maxtemp                 9.09
mintemp                 9.09
moon_illumination       9.09
moon_phase              9.09
precip                  9.09
pressure                9.09
sunhour                 9.09
totalsnow               9.09
uv_index                9.09
visibility              9.09
weather_code            9.09
weather_descriptions    9.09
wind_degree             9.09
wind_dir                9.09
wind_speed              9.09
windchill               9.09
windgust                9.09
dtype: float64


In [19]:
w.nunique()

date                    3652
avg_uv_index               1
avgtemp                   43
cloudcover               101
dewpoint                  42
feelslike                 54
heatindex                 49
humidity                  59
maxtemp                   45
mintemp                   42
moon_illumination        101
moon_phase                 8
precip                    11
pressure                  60
sunhour                   93
totalsnow                 40
uv_index                   1
visibility                11
weather_code              17
weather_descriptions      17
wind_degree              357
wind_dir                  16
wind_speed                32
windchill                 50
windgust                  51
dtype: int64